# Sesión 8 · Del notebook a una página pública

Un notebook lo abre quien tiene Python. Tu jefe, un regidor o un periodista
no lo van a abrir nunca.

**Streamlit** convierte el mismo código en una página web con filtros. Al
final de esta clase vas a tener una **URL que puedes mandar por WhatsApp**.

La diferencia con todo lo anterior: esto no se escribe en un notebook, se
escribe en un archivo `.py`. Aquí vemos las piezas; el archivo completo está
en `app.py`, al lado de este notebook.

In [1]:
import pandas as pd

datos = pd.read_csv("datos.csv", dtype={"ubigeo": str})
datos.head()

,ubigeo,departamento,provincia,distrito,casos_dengue,n_establecimientos
0,140106,LAMBAYEQUE,CHICLAYO,LA VICTORIA,182.0,6.0
1,140107,LAMBAYEQUE,CHICLAYO,LAGUNAS,1.0,6.0
2,140108,LAMBAYEQUE,CHICLAYO,MONSEFU,12.0,4.0
3,140115,LAMBAYEQUE,CHICLAYO,SAÑA,233.0,5.0
4,140116,LAMBAYEQUE,CHICLAYO,CAYALTI,74.0,3.0


## 1. La idea completa en 5 líneas

Un archivo llamado `hola.py` con esto adentro:

```python
import streamlit as st
import pandas as pd

st.title("Mi primer dashboard")
datos = pd.read_csv("datos.csv")
st.dataframe(datos)
```

Y se corre desde la terminal:

```bash
uv run streamlit run hola.py
```

Se abre solo en el navegador. **Cada vez que guardas el archivo, la página se
actualiza.**

## 2. Las piezas que vas a usar

### Texto

```python
st.title("Título grande")
st.header("Sección")
st.subheader("Subsección")
st.write("Texto normal, y también acepta tablas y gráficos")
st.caption("Letra chica, para la fuente")
```

### Mostrar datos

```python
st.dataframe(df)              # tabla interactiva, se ordena con clic
st.table(df.head())           # tabla estática
st.metric("Casos totales", "231,122")
```

### Filtros (widgets)

Cada widget **devuelve** lo que el usuario eligió. Esa es toda la magia:

```python
depto = st.selectbox("Departamento", ["Piura", "Loreto"])
deptos = st.multiselect("Departamentos", lista, default=lista[:5])
minimo = st.slider("Casos mínimos", 0, 1000, 100)
texto  = st.text_input("Buscar distrito")
```

Y se usa igual que cualquier variable de pandas:

```python
filtrado = datos[datos["departamento"] == depto]
```

Cuando el usuario mueve el filtro, **Streamlit vuelve a correr todo el
archivo de arriba abajo** con el nuevo valor. Por eso se escribe de corrido,
sin bucles ni botones de "actualizar".

### Layout

```python
st.sidebar.slider(...)        # lo manda a la barra lateral

c1, c2, c3 = st.columns(3)    # tres columnas
c1.metric("Casos", "18,534")

with c2:
    st.plotly_chart(fig)
```

### Gráficos y mapas

Los mismos de la sesión 6 y 7, sin cambiar nada:

```python
st.plotly_chart(fig, width="stretch")
```

## 3. La única trampa: `@st.cache_data`

Streamlit vuelve a correr **todo** el archivo cada vez que alguien toca un
filtro. Si lees un archivo pesado en cada corrida, la app se arrastra.

```python
@st.cache_data
def cargar():
    return pd.read_csv("datos.csv")

datos = cargar()
```

Con eso, el archivo se lee una sola vez y el resultado queda guardado. Esta
línea es la diferencia entre una app usable y una inservible.

## 4. Probemos la lógica aquí antes de llevarla a la app

Esto es lo que hará el filtro de la barra lateral.

In [2]:
elegidos = ["Piura", "Tumbes", "Loreto"]
minimo = 100

filtrado = datos[
    datos["departamento"].str.title().isin(elegidos) & (datos["casos_dengue"] >= minimo)
]
filtrado.shape

(67, 6)

### Y esto lo que mostrarán las métricas de arriba

In [3]:
print("Distritos:", len(filtrado))
print("Casos totales:", int(filtrado["casos_dengue"].sum()))
print("Establecimientos:", int(filtrado["n_establecimientos"].sum()))
print("Casos por establecimiento:",
      round(filtrado["casos_dengue"].sum() / filtrado["n_establecimientos"].sum(), 1))

Distritos: 67
Casos totales: 106282
Establecimientos: 522
Casos por establecimiento: 203.6


## 5. Correr la app

Abre una terminal, entra a esta carpeta y ejecuta:

```bash
uv run streamlit run app.py
```

Se abre en `http://localhost:8501`. Mientras corre, la terminal queda ocupada;
se detiene con `Ctrl+C`.

Abre `app.py` al lado y lee el archivo completo: son 130 líneas y no hay nada
que no hayas visto en las sesiones anteriores.

## 6. Publicarlo: Streamlit Community Cloud

Es gratis y usa la cuenta de GitHub que creaste en la sesión 1.

### Paso 1 · Tu carpeta necesita 3 cosas

```
mi-dashboard/
├── app.py             # el código
├── datos.csv          # los datos (deben estar EN el repo)
└── requirements.txt   # las librerías
```

El `requirements.txt` de esta app:

```
streamlit
pandas
plotly
geopandas
```

### Paso 2 · Súbelo a GitHub

```bash
git add .
git commit -m "Mi dashboard"
git push
```

El repositorio tiene que ser **público**.

### Paso 3 · Conecta

1. Entra a [share.streamlit.io](https://share.streamlit.io)
2. *Sign in with GitHub*
3. *New app* → elige tu repositorio → archivo `app.py`
4. *Deploy*

En dos o tres minutos tienes tu URL:
`https://tu-usuario-mi-dashboard.streamlit.app`

### Los tres errores que verás

| Error | Causa | Solución |
|---|---|---|
| `ModuleNotFoundError` | falta la librería en `requirements.txt` | agrégala y haz push |
| `FileNotFoundError` | el CSV no está en el repo (¿lo bloqueó `.gitignore`?) | súbelo, o léelo por URL |
| La app se queda cargando | archivo muy pesado, o falta `@st.cache_data` | reduce los datos y cachea |

Los datos tienen que vivir **dentro del repositorio**. Una ruta como
`../../data/archivo.csv` funciona en tu computadora y falla en la nube.

---
## Lo que hicimos

| Para | Código |
|---|---|
| Título | `st.title()`, `st.header()` |
| Tabla | `st.dataframe(df)` |
| Número grande | `st.metric(label, valor)` |
| Filtros | `st.selectbox()`, `st.multiselect()`, `st.slider()` |
| Barra lateral | `st.sidebar.<lo que sea>` |
| Columnas | `c1, c2 = st.columns(2)` |
| Gráfico | `st.plotly_chart(fig, width="stretch")` |
| No releer archivos | `@st.cache_data` |
| Descargar | `st.download_button()` |
| Correr | `uv run streamlit run app.py` |

---
## Trabajo final

Está en `assignments/tarea-3-dashboard.md`. Entregas **una URL**, no un archivo.